In [5]:
import polars as pl
from pathlib import Path
from urllib.request import urlopen, Request
from tqdm import tqdm

In [ ]:

out_dir = Path("/mnt/Fast2T/data/common_crawl/")

urls = (
    pl.read_csv("/mnt/Fast2T/kotlin/2026/data-science-project/data/paths")
    .with_columns(url="https://data.commoncrawl.org/" + pl.col("url"))
    .sample(100)
    .get_column("url")
    .to_list()
)
CHUNK = 1 << 20

for i, url in enumerate(urls):
    dest = out_dir / Path(url).name
    if dest.exists():
        continue

    resp = urlopen(Request(url))
    total = int(resp.headers.get("Content-Length", 0)) or None

    with (
        open(dest, "wb") as f,
        tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=f"[{i + 1}/{len(urls)}] {dest.name[:40]}",
        ) as bar,
    ):
        while chunk := resp.read(CHUNK):
            f.write(chunk)
            bar.update(len(chunk))

In [ ]:
import gzip
import re
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq
from warcio.archiveiterator import ArchiveIterator

_SURROGATES = re.compile(f"[{chr(0xD800)}-{chr(0xDFFF)}]")


def sanitize_utf8(s: str) -> str:
    if not s:
        return s
    s = _SURROGATES.sub("\ufffd", s)
    return s.encode("utf-8", "replace").decode("utf-8")


_charset_re = re.compile(r"charset\s*=\s*([^\s;]+)", re.IGNORECASE)


def charset_from_content_type(ct: str) -> str | None:
    if not ct:
        return None
    m = _charset_re.search(ct)
    return m.group(1).strip("\"'") if m else None


def decode_html(body: bytes, charset: str | None) -> str:
    if not body:
        return ""
    enc = (charset or "utf-8").strip()
    try:
        text = body.decode(enc, errors="replace")
    except (LookupError, UnicodeDecodeError):
        text = body.decode("utf-8", errors="replace")
    return sanitize_utf8(text)


def warc_to_single_parquet_html_only(
        warc_paths: list[Path],
        out_file: Path,
        chunk_size: int = 20_000,
        zstd_level: int | None = None,  # optional
        require_status_200: bool = False  # optional
) -> None:
    out_file.parent.mkdir(parents=True, exist_ok=True)
    if out_file.exists():
        out_file.unlink()  # avoid accidental append/overwrite ambiguity

    contents: list[str] = []
    writer: pq.ParquetWriter | None = None

    def flush() -> None:
        nonlocal contents, writer
        if not contents:
            return

        n = len(contents)
        df = df = pl.DataFrame(
            {
                "content": pl.Series("content", [sanitize_utf8(x) for x in contents], dtype=pl.Utf8),
            }
        )

        table = df.to_arrow()

        if writer is None:
            writer = pq.ParquetWriter(
                where=str(out_file),
                schema=table.schema,
                compression="zstd",
                compression_level=zstd_level,
                use_dictionary=True,
                write_statistics=False
            )

        writer.write_table(table)
        contents = []

    try:
        for warc_path in warc_paths:
            with gzip.open(warc_path, "rb") as f:
                for record in ArchiveIterator(f):
                    if getattr(record, "rec_type", None) != "response":
                        continue

                    hh = getattr(record, "http_headers", None)
                    if not hh:
                        continue

                    ct = (hh.get_header("Content-Type") or "")
                    if "text/html" not in ct.lower():
                        continue

                    if require_status_200:
                        try:
                            if int(hh.get_statuscode() or 0) != 200:
                                continue
                        except Exception:
                            continue

                    body_bytes = record.content_stream().read()
                    html = decode_html(body_bytes, charset_from_content_type(ct))
                    if not html:
                        continue

                    contents.append(sanitize_utf8(html))
                    if len(contents) >= chunk_size:
                        flush()

        flush()
    finally:
        if writer is not None:
            writer.close()


for file in Path("/mnt/data-dump/common_crawl/").iterdir():
    if not file.name.endswith(".warc.gz"):
        continue

    print(file)
    warc_to_single_parquet_html_only(
        warc_paths=[file],
        out_file=Path("/mnt/Fast2T/data/common_data_parquet/") / file.name.replace(".warc.gz", ".parquet"),
    )

In [2]:
import polars as pl
import pyarrow.parquet as pq
from pathlib import Path
import tqdm

files = Path("/mnt/Fast2T/data/new/processed").glob("*.parquet")
rows = [pq.read_metadata(f).num_rows for f in tqdm.tqdm(files)]

print(rows)
print(sum(rows))
print(sum(rows) / len(rows))

1153it [00:03, 310.10it/s]

[10291, 8669, 10508, 10618, 10496, 10407, 10478, 7931, 11223, 10367, 10426, 10501, 8843, 11037, 10961, 10306, 10408, 10515, 10569, 10463, 10828, 11119, 10135, 10323, 10784, 10750, 8784, 10478, 11082, 10276, 10959, 10426, 10571, 10643, 11002, 10641, 10126, 10377, 10628, 10440, 10610, 10410, 10484, 10376, 10427, 10611, 11051, 10366, 10375, 10438, 9989, 10485, 11047, 10665, 10491, 10560, 10575, 10384, 10259, 10440, 10283, 11168, 10863, 10984, 11296, 10568, 10491, 10514, 10280, 11220, 11115, 10582, 10390, 10307, 10663, 10711, 10588, 10894, 10546, 10570, 10617, 10069, 10407, 10654, 8568, 10222, 10486, 10604, 10966, 11008, 10347, 10467, 10671, 10623, 10228, 10281, 10416, 10989, 10501, 10601, 10335, 10366, 10159, 10392, 10506, 10357, 10290, 11031, 10597, 8656, 10356, 10416, 10526, 10570, 10406, 10389, 10315, 10084, 10576, 10341, 10346, 10357, 10488, 10609, 10506, 10414, 10474, 7600, 10503, 10570, 10935, 10454, 11057, 10430, 10341, 10409, 11088, 10424, 10659, 10281, 10518, 7882, 11148, 11273, 

In [16]:
import polars as pl

ai_type = pl.Enum(["AI", "UNKNOWN", "HUMAN"])

blacklist = [
    "Please wait while your request is being verified...",
    "AI scrapers break the web, to use this page you'll need JavaScript enabled.",
    "Javascript is required for this site. Consider enabling Javascript or upgrading to a modern browser.",
    "This domain name registration has expired and renewal or deletion are pending. If you are the registrant and want to renew the domain name, please contact your registration service provider.",
    "We're sorry but doesn't work properly without JavaScript enabled. Please enable it to continue."
]

(
    pl
    .read_parquet("/mnt/Fast2T/data/new/clean_data/CC-MAIN-20260212024315-20260212054315-00248.parquet")
    .with_columns(
        pl
        .when(pl.col('ai') < 0.005).then(pl.lit("HUMAN"))
        .when(pl.col('ai') > 0.9).then(pl.lit("AI"))
        .otherwise(pl.lit("UNKNOWN"))
        .cast(ai_type).alias("is_ai"),
        pl
        .col("outflow")
        .map_elements(
            lambda s: [{ "url": k, "occurrences": v } for k, v in s.items() if v is not None],
            return_dtype=pl.List(pl.Struct({ "url": pl.String, "occurrences": pl.Int64 })),
        )
    )
    .filter(pl.col("lang_en") > 0.3)
    .filter(~pl.col("text").str.contains_any(blacklist))
    .drop("lang_en", "ai")
)

root_host,host,url,text,outflow,is_ai
str,str,str,str,list[struct[2]],enum
"""1812pizzacompany.com""","""1812pizzacompany.com""","""http://1812pizzacompany.com/po…","""623-B West St Hwy 18Manila, AR…","[{""www.facebook.com"",2}, {""1812pizzacompany.com"",11}, … {""aceonetechnologies.com"",1}]","""HUMAN"""
"""zrxdjt.com""","""2017.zrxdjt.com""","""http://2017.zrxdjt.com/index.p…","""Group Introduction Corporate C…","[{""2017.zrxdjt.com"",59}, {""ebs.zrxdjt.com"",2}, … {""beian.miit.gov.cn"",1}]","""HUMAN"""
"""edu.pe.ca""","""356.schoollibrary.edu.pe.ca""","""http://356.schoollibrary.edu.p…","""No Result found ! No results m…","[{""356.schoollibrary.edu.pe.ca"",3}]","""HUMAN"""
"""blogspot.com""","""4christum.blogspot.com""","""http://4christum.blogspot.com/…","""Bergoglio's Disney Immorality …","[{""www.youtube.com"",2}, {""twitter.com"",1}, … {""en-denzingerbergoglio.com"",1}]","""HUMAN"""
"""acccontern.lu""","""acccontern.lu""","""http://acccontern.lu/English/v…","""Note: ""This site contains link…","[{""acccontern.lu"",23}, {""www.ost.lu"",1}, … {""www.comat.lu"",1}]","""HUMAN"""
…,…,…,…,…,…
"""zoom.earth""","""zoom.earth""","""https://zoom.earth/places/braz…","""Weather forecasts, rain radar,…",[],"""HUMAN"""
"""rotary2000.ch""","""zuerich-oberland.rotary2000.ch""","""https://zuerich-oberland.rotar…","""German English French Italian …",[],"""HUMAN"""
"""zib.de""","""zuse.zib.de""","""https://zuse.zib.de/collection…","""Skip to main menu Skip to cont…",[],"""HUMAN"""
